# 伪随机数的预测 ― 用 Z3 反推 `rand()` 的种子

## 本章要点

伪随机数看起来是随机的，但只要知道生成机制，就能预测后面的值。本章以 Visual C++ 的 `rand()` 为例。

- `rand()` 是用线性同余生成器（LCG）做的
- 只要知道几个输出，就能用 Z3 反推出内部状态（种子）
- 从反推出的种子，预测下一个值

主角是 `BitVec`（固定长度的比特向量）和 `URem`（无符号取余）。

## 线性同余生成器（LCG）

**线性同余生成器**（Linear Congruential Generator, LCG）是生成伪随机数的方法之一。它用下面的递推式生成下一个状态。

$$
X_{n+1} = \left(A \times X_n + B\right) \bmod M
$$

$A$, $B$, $M$ 是常数，通常满足下面的条件。

- $M > A$
- $M > B$
- $A > 0$
- $B \geq 0$

如果 $M = 2^{32}$，那正好就是 32 位整数的乘法和加法里“把溢出的部分丢掉”的计算。

## Visual C++ 的 `rand()`

Visual C++ 的 `rand()` 使用线性同余生成器。源码如下。

```c
static long holdrand = 1L;

int __cdecl rand(void)
{
    return (((holdrand = holdrand * 214013L + 2531011L) >> 16) & 0x7FFF);
}

void __cdecl srand(unsigned int seed)
{
    holdrand = (long)seed;
}
```

它做两件事。

1. 更新内部状态 `holdrand`

$$
\text{holdrand} = (\text{holdrand} \times 214013 + 2531011) \bmod 2^{32}
$$

2. 取出高 16 位，作为 0～32767（`0x7FFF`）的整数返回

输出只是内部状态的**一部分**。正因为如此，从输出反推状态就成了“信息够不够”的问题。

我们准备一个行为相同的 Python 类。

In [1]:
class PseudoRandom:
    """和 Visual C++ 的 rand() 行为相同的伪随机数"""

    def __init__(self, seed=1):
        self._holdrand = seed

    def srand(self, seed: int):
        self._holdrand = seed

    def rand(self) -> int:
        self._holdrand = (self._holdrand * 214013 + 2531011) & 0xFFFFFFFF
        return (self._holdrand >> 16) & 0x7FFF

## 生成 8 个随机数

用种子 `1792` 生成 8 个 `rand() % 100`（0～99）。第 9 个留着核对答案。

In [2]:
rand = PseudoRandom(1792)
numbers = [rand.rand() % 100 for _ in range(8)]
print(numbers)
print(rand.rand() % 100)

[90, 79, 97, 7, 20, 77, 17, 89]
16


## 用 Z3 反推种子

只看 `numbers` 里的 8 个数，能猜出第 9 个吗？

内部状态是 32 位，所以设成 `BitVec('seed', 32)` 这样的变量。然后把 `rand()` 的计算原样写成约束条件。

1. `states[0]` 是“种子前进一次后的状态”，`states[1]` 是再下一个……依次排列
2. 相邻状态满足 `state_{i+1} = state_i * a + b`（在 32 位里循环，所以乘法溢出会自动丢掉）
3. 从每个状态取出的输出 `(state >> 16) & 0x7FFF` 除以 `mod` 的余数，要和观测到的 `numbers` 一致

因为我们只知道除以 `mod` 后的余数，状态本身不一定唯一确定。即便如此，也能找到“生成同一数列的种子”。

In [3]:
from z3 import *


def find_seed_and_next(numbers, mod, a=214013, b=2531011):
    """从观测到的输出序列，求出种子和下一个输出"""
    seed = BitVec('seed', 32)
    states = [BitVec(f's_{i}', 32) for i in range(len(numbers) + 1)]
    outputs = [URem((s >> 16) & 0x7FFF, mod) for s in states]

    solver = Solver()

    # 1. 状态的转移（LCG）
    for prev, cur in zip([seed] + states[:-1], states):
        solver.add(prev * a + b == cur)

    # 2. 和观测到的输出一致
    for out, observed in zip(outputs, numbers):
        solver.add(out == observed)

    if solver.check() != sat:
        return None

    model = solver.model()
    return model[seed].as_long(), model.eval(outputs[-1]).as_long()


seed_val, next_val = find_seed_and_next(numbers, mod=100)
print(f'seed = {seed_val}')
print(f'next = {next_val}')

seed = 1792
next = 16


找到了原来的种子 `1792`，下一个值也预测为 `16`。实际的第 9 个也是 `16`（和上面格子的输出对比一下）。

用找到的种子重新生成一遍数列。

In [4]:
rand = PseudoRandom(seed_val)
[rand.rand() % 100 for _ in range(9)]

[90, 79, 97, 7, 20, 77, 17, 89, 16]

完全相同的 9 个数排列出来了。预测成功。

不过，因为用 `mod 100` 做了截断，**种子候选不一定只有一个**。这个问题里 `1792` 和 `2147485440` 两个都满足条件，两者都会得到同样的数列。输出信息越少，候选就越多。

## 应用：用骰子连续掷出 10 次 6

用 `rand() % 6 + 1` 掷骰子时，来找一个“连续 10 次都是 6”的种子。

要掷出 6，只要 `rand() % 6 == 5`，所以把 `[5] * 10` 作为观测值传进去。

In [5]:
seed_val, next_val = find_seed_and_next([5] * 10, mod=6)
print(seed_val)

rand = PseudoRandom(seed_val)
[rand.rand() % 6 for _ in range(11)]

3689596556


[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 0]

找到了连续 10 次都是 `5`（＝骰子掷出 6）的种子。

不过，放进条件的只是**前 10 次**。第 11 次没有放进条件，所以不一定还是 `5`（这次执行是 `0`）。如果想“连续 11 次都是 6”，把 `[5] * 11` 这 11 次都放进条件就好。

另外，满足条件的种子不止一个，每次执行可能会找到不同的种子。

## `BitVec` 和 `URem`

`BitVec` 是 Z3 的**固定长度比特向量**类型。

- `BitVec('x', 32)` 当作“32 位整数”使用
- 乘法和加法溢出的比特会被丢掉（和 C 语言的无符号整数一样）
- 可以使用 `&`, `|`, `^`, `>>`, `<<` 等位运算

`URem(a, b)` 是**无符号取余**（`a % b`）。`BitVec` 没有符号，所以要用 `URem()` 代替 `%`。

有了这两个，`rand()` 的代码可以原样照抄。

| C 的代码 | Z3 的表达式 |
|---|---|
| `holdrand * 214013 + 2531011` | `prev * 214013 + 2531011` |
| 32 位溢出的部分丢掉 | `BitVec` 的计算（自动） |
| `holdrand >> 16` | `s >> 16` |
| `& 0x7FFF` | `& 0x7FFF` |
| `% mod` | `URem(..., mod)` |

## 小结

- 只要知道随机数的生成式，就能从输出反推内部状态（把生成式原样写进 `BitVec`）
- 如果输出只是状态的一部分，种子候选可能不止一个
- 用反推出的种子，可以预测下一个值
- 像“连续 10 次都是 6”这样的条件，也只要作为观测值传进去就能搜索种子